In [4]:
import numpy as np
import pandas as pd
import subprocess
import re
import seaborn as sns
import matplotlib.pyplot as plt
import io
import os

def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

In [ ]:
%%bash
gunzip /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna.gz

In [2]:
!cat /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references.txt

adig /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/adig.fa
apal /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna


In [10]:
!cat /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/cactus_align.sh

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks=40
#SBATCH --mem=256GB
#SBATCH --time=48:00:00
#SBATCH --partition=mpcs_hifmb.p
#SBATCH --job-name=cactus

source ~/.bashrc
conda activate cactus

mkdir -p /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/liftover_process
cd /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/liftover_process
export PATH=/fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/cactus-bin-v2.9.0/bin/:$PATH

cactus \
	jobStore \
	../references.txt \
	adig_apal.hal \
	--defaultMemory 250G \
	--defaultCores 40 \
	--binariesMode local \
	--workDir /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/liftover_process


In [9]:
!sbatch /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/cactus_align.sh

Submitted batch job 9848439


In [5]:
!cat /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references.txt

adig /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/adig.fa
apal /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna


In [2]:
%%bash
source ~/.bashrc
conda activate cactus
cd /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/liftover_process
mkdir -p ./chains
export PATH="/fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/cactus-bin-v2.9.0/bin:$PATH"
cactus-hal2chains \
    ./chains/jobStore adig_apal.hal \
    --defaultMemory 64G \
    --defaultCores 8 \
    --binariesMode local \
    ./chains --queryGenomes apal

/user/noge4093/miniconda3/envs/cactus/lib/python3.12/site-packages/htcondor/__init__.py:49: UserWarning: Neither the environment variable CONDOR_CONFIG, /etc/condor/, /usr/local/etc/, nor ~condor/ contain a condor_config source. Therefore, we are using a null condor_config.
  _warnings.warn(message)
[2025-08-12T21:33:42+0200] [MainThread] [I] [toil.statsAndLogging] Enabling realtime logging in Toil
[2025-08-12T21:33:42+0200] [MainThread] [W] [toil.lib.humanize] Deprecated toil method.  Please use "toil.lib.conversions.human2bytes()" instead."
[2025-08-12T21:33:42+0200] [MainThread] [W] [toil.lib.humanize] Deprecated toil method.  Please use "toil.lib.conversions.human2bytes()" instead."
[2025-08-12T21:33:42+0200] [MainThread] [I] [toil.statsAndLogging] Cactus Command: /user/noge4093/miniconda3/envs/cactus/bin/cactus-hal2chains ./chains/jobStore adig_apal.hal --defaultMemory 64G --defaultCores 8 --binariesMode local ./chains --queryGenomes apal
[2025-08-12T21:33:42+0200] [MainThread] [I

In [6]:
%%bash
source ~/.bashrc
conda activate gatk
gatk CreateSequenceDictionary \
    R=/fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna \
    O=/fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna.dict

Using GATK jar /fs/dss/home/noge4093/miniconda3/envs/gatk/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /fs/dss/home/noge4093/miniconda3/envs/gatk/share/gatk4-4.6.2.0-0/gatk-package-4.6.2.0-local.jar CreateSequenceDictionary R=/fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna O=/fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/references/GCF_964030605.1_jaAcrPala1.3_genomic.fna.dict
INFO	2025-08-12 21:47:34	CreateSequenceDictionary	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** 
https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this

Tool returned:
0


In [7]:
!gunzip --keep /fs/dss/groups/agmarinecons/apal_reference_panel/apal_reference_panel/liftover_process/chains/adig_vs_apal.chain.gz